In [1]:
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.compose import make_column_transformer
from sklearn.model_selection import train_test_split

from tensorflow import keras
from tensorflow.keras import layers

home_data = pd.read_csv(
    '/kaggle/input/datasets/girishchinnaiya/trainingdata/train.csv'
)

# Target
y = home_data.SalePrice

# Features
features = [
    'LotArea',
    'YearBuilt',
    '1stFlrSF',
    '2ndFlrSF',
    'FullBath',
    'BedroomAbvGr',
    'TotRmsAbvGrd'
]

X = home_data[features]

# Train-validation split
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Preprocessing
preprocessor = make_column_transformer(
    (StandardScaler(), features),
)

X_train = preprocessor.fit_transform(X_train)
X_valid = preprocessor.transform(X_valid)

# Input shape
input_shape = [X_train.shape[1]]

# Model
model = keras.Sequential([
    layers.BatchNormalization(input_shape=input_shape),

    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(1)
])

model.compile(
    optimizer='adam',
    loss='mae'
)

early_stopping = keras.callbacks.EarlyStopping(
    patience=5,
    min_delta=0.001,
    restore_best_weights=True,
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_valid, y_valid),
    batch_size=512,
    epochs=200,
    callbacks=[early_stopping],
)

test_data = pd.read_csv(
    '/kaggle/input/competitions/home-data-for-ml-course/test.csv'
)

X_test = test_data[features]

X_test = preprocessor.transform(X_test)

predictions = model.predict(X_test)

predictions = predictions.flatten()

2026-05-07 10:27:57.426049: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778149677.717178      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778149677.800118      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778149678.522938      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778149678.523012      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778149678.523015      16 computation_placer.cc:177] computation placer alr

Epoch 1/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 183ms/step - loss: 181518.4688 - val_loss: 178839.8281
Epoch 2/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - loss: 179963.8281 - val_loss: 178839.7344
Epoch 3/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - loss: 181278.7188 - val_loss: 178839.6562
Epoch 4/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - loss: 181313.3906 - val_loss: 178839.5781
Epoch 5/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - loss: 180907.4844 - val_loss: 178839.5000
Epoch 6/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - loss: 182991.5156 - val_loss: 178839.4062
Epoch 7/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - loss: 181252.9062 - val_loss: 178839.3125
Epoch 8/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - loss: 182037.6094 - val_loss: 178839.2344
Epoch 9/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - loss: 180213.6250 - val_loss: 178839.1562
Epoch 10/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - loss: 182274.1406 - val_loss: 178839.0469
Epoch 11/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - loss:

In [2]:
output = pd.DataFrame({
    'Id': test_data.Id,
    'SalePrice': predictions
})
output.to_csv('submission.csv', index=False)